In [47]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import dill

In [48]:
class BPE:
    def __init__(self, vocab_size: int):
        self.vocab_size = vocab_size
        self.id2token = None
        self.token2id = None
    
    def fit(self, text: str):
        uniq_tokens = sorted(set(text))
        tokens_list = list(text)

        while len(uniq_tokens) < self.vocab_size:
            print(len(uniq_tokens), '|', self.vocab_size)
            pairs = {}

            for i in range(len(tokens_list)-1):
                if (tokens_list[i], tokens_list[i+1]) not in pairs:
                    pairs[(tokens_list[i], tokens_list[i+1])] = 0
                pairs[(tokens_list[i], tokens_list[i+1])] += 1
            
            if len(pairs) == 0: break

            top_freq = 0
            top_pair = None
            for pair, freq in pairs.items():
                if freq > top_freq:
                    top_freq = freq
                    top_pair = pair
            
            uniq_tokens.append(top_pair[0]+top_pair[1])

            upd_tokens_list = []
            i=0
            while i < len(tokens_list):
                if (i < len(tokens_list) - 1 and tokens_list[i:i+2] == list(top_pair)):
                    upd_tokens_list.append(top_pair[0]+top_pair[1])
                    i+=2
                else:
                    upd_tokens_list.append(tokens_list[i])
                    i+=1

            tokens_list = upd_tokens_list
        
        self.id2token = {i: token for i, token in enumerate(uniq_tokens[:self.vocab_size])}
        self.token2id = {token: i for i, token in self.id2token.items()}

    def encode(self, text: str) -> list[int]:

        tokens_list = list(text)

        fin_flag = True
        while fin_flag:
            fin_flag = False

            upd_tokens_list = []
            i=0
            while i < len(tokens_list):
                if (i < len(tokens_list) - 1 and tokens_list[i]+tokens_list[i+1] in self.token2id):
                    upd_tokens_list.append(tokens_list[i]+tokens_list[i+1])
                    i+=2
                    fin_flag = True
                else:
                    upd_tokens_list.append(tokens_list[i])
                    i+=1
            
            tokens_list = upd_tokens_list
        
        token_id_list = []

        for token in tokens_list:
            token_id_list.append(self.token2id[token])
        
        return(token_id_list)
    
    def decode(self, token_ids: list[int]) -> str:

        return ''.join(self.id2token[i] for i in token_ids)

    def save(self, filename):
        with open(filename, 'wb') as f:
            dill.dump(self, f)
        print(f"Объект сохранён в {filename}")

    @classmethod
    def load(cls, filename):
        with open(filename, 'rb') as f:
            obj = dill.load(f)
                
        print(f"Объект загружен из {filename}")
        return obj


In [49]:
drova = "На дворе дрова, за двором дрова, дрова вширь двора, не вместит двор дров, надо дрова выдворить на дровяной двор."

In [50]:
bpe = BPE(28)
bpe.fit(drova)

print(bpe.id2token)
print(bpe.encode('вором дрова, дрова вширь двора, не вмест'))
print(bpe.decode(bpe.encode('вором дрова, дрова вширь двора, не вмест')))

21 | 28
22 | 28
23 | 28
24 | 28
25 | 28
26 | 28
27 | 28
{0: ' ', 1: ',', 2: '.', 3: 'Н', 4: 'а', 5: 'в', 6: 'д', 7: 'е', 8: 'з', 9: 'и', 10: 'й', 11: 'м', 12: 'н', 13: 'о', 14: 'р', 15: 'с', 16: 'т', 17: 'ш', 18: 'ы', 19: 'ь', 20: 'я', 21: ' д', 22: 'ро', 23: 'во', 24: ' дро', 25: ' дров', 26: ' дво', 27: ' двор'}
[23, 22, 11, 25, 4, 1, 25, 4, 0, 5, 17, 9, 14, 19, 27, 4, 1, 0, 12, 7, 0, 5, 11, 7, 15, 16]
вором дрова, дрова вширь двора, не вмест


In [51]:
class TokenEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, emb_size: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_size = emb_size

        self.emb_matrix = nn.Embedding(vocab_size, emb_size)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        
        return self.emb_matrix(x)

In [118]:
class PositionalEmbeddings(nn.Module):
    def __init__(self, max_seq_len: int, emb_size: int):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.emb_size = emb_size

        self.emb_matrix = nn.Embedding(max_seq_len, emb_size)
    
    def forward(self, seq_len: torch.Tensor) -> torch.Tensor:

        return self.emb_matrix(torch.arange(0, seq_len, device='cpu'))

In [53]:
class HeadAttention(nn.Module):
    def __init__(self, emb_size: int, head_size: int, max_seq_len: int):
        super().__init__()
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.w_k = nn.Linear(self.emb_size, self.head_size)
        self.w_q = nn.Linear(self.emb_size, self.head_size)
        self.w_v = nn.Linear(self.emb_size, self.head_size)
        self.mask_attention = torch.tril(torch.ones(max_seq_len, max_seq_len))
    
    def forward(self, x: float):
        seq_len = x.shape[1]

        self.attention_matrix = torch.matmul(self.w_q(x), self.w_k(x).transpose(1, 2))
        self.attention_matrix /= np.sqrt(self.head_size)
        self.mask_matrix = self.mask_attention[:seq_len, :seq_len]

        self.attention_matrix = torch.where(self.mask_matrix == 1,
                                            self.attention_matrix,
                                            torch.tensor(float('-inf'), device=self.attention_matrix.device, dtype=self.attention_matrix.dtype))
        
        self.attention_matrix = torch.softmax(self.attention_matrix, dim=2)

        return torch.matmul(self.attention_matrix, self.w_v(x))

In [54]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads: int, emb_size: int, head_size: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len

        self.heads = nn.ModuleList([HeadAttention(emb_size, head_size, max_seq_len)] * num_heads)
        self.linear = nn.Linear(self.head_size * self.num_heads, self.emb_size)
        self.drop = torch.nn.Dropout(p=dropout)
    
    def forward(self, x):

        outp_heads = []
        for head in self.heads:
            outp_heads.append(head(x))
        
        conc = torch.cat(outp_heads, dim=2)
        linear_res = self.linear(conc)
        
        return self.drop(linear_res)

In [55]:
class FeedForward(nn.Module):
    def __init__(self, emb_size: int, dropout: float = 0.1):
        super().__init__()
        self.emb_size = emb_size
        self.dropout = dropout

        self.lin1 = nn.Linear(emb_size, emb_size*4)
        self.relu1 = nn.ReLU()
        self.lin2 = nn.Linear(emb_size*4, emb_size)
        self.drop = torch.nn.Dropout(p=dropout)
    
    def forward(self, x):

        return(self.drop(self.lin2(self.relu1(self.lin1(x)))))

In [56]:
class Decoder(nn.Module):
    def __init__(self, num_heads:int, emb_size:int, head_size:int, max_seq_len: int, dropout: float=0.1):
        super().__init__()
        self.multi_head = MultiHeadAttention(num_heads, emb_size, head_size, max_seq_len, dropout)
        self.feed_forward = FeedForward(emb_size, dropout)

        self.norm1 = nn.LayerNorm(emb_size)
        self.norm2 = nn.LayerNorm(emb_size)
    
    def forward(self, x):
        x_proc = self.multi_head(x)
        x_proc += x
        x_proc = self.norm1(x_proc)
        x_proc = self.feed_forward(x_proc)
        x_proc += x
        x_proc = self.norm2(x)

        return x_proc

In [92]:
class GPT(nn.Module):
    def __init__(self, vocab_size: int, max_seq_len: int, emb_size: int, num_heads: int, head_size: int, num_layers: int, dropout: float=0.1, device: str='cuda'):
        super().__init__()

        self.token_emb = TokenEmbeddings(vocab_size, emb_size)
        self.pos_emb = PositionalEmbeddings(max_seq_len, emb_size)
        self.drop = nn.Dropout(dropout)

        self.decoders = nn.Sequential([Decoder(num_heads, emb_size, head_size, max_seq_len, dropout)] * num_layers)

        self.lin = nn.Linear(emb_size, vocab_size)

    def forward(self, x):
        x_embs = self.token_emb(x) + self.pos_emb(x.shape[1])
        x_dec = self.decoders(self.drop(x_embs))

        return self.lin(x_dec)
    
    def generate(self, x, max_new_tokens: int, do_sample: bool, temperature: float=1.0, top_k: int = None, top_p: float = None):
        for i in range(max_new_tokens):
            tokens = x[:, -self.max_seq_len:]
            logits = self.forward(tokens)[:, -1, :] / temperature

            if top_k and do_sample:
                last_logits = logits[:, -1, :]
                top_k_vals, _ = torch.sort(last_logits, dim=1, descending=True)
                min_value = top_k_vals[:, top_k-1].unsqueeze(1)
                last_logits[last_logits < min_value] = -float('inf')
            
            if top_p and do_sample:
                last_logits = logits[:, -1, :]
                probs = torch.softmax(last_logits, dim=1)
                probs_sort, idx_sort = torch.sort(probs, dim=1, descending=True)
                max_idx = idx_sort[:, 0].unsqueeze(1)
                arg_idx = torch.argsort(idx_sort, dim=1)
                cumsum = torch.cumsum(probs_sort, dim=1)
                cumsum_probs = torch.gather(cumsum, dim=1, index=arg_idx)
                mask = cumsum_probs > top_p
                mask.scatter_(dim=1, index=max_idx, value=False)
                last_logits[mask] = -float('inf')

            probs = torch.softmax(logits[:, -1, :], dim=1)

            if not do_sample:
                arg_log = torch.torch.argmax(probs, dim=1)
            else:
                arg_log = torch.multinomial(probs, 1)

            logits_upd = torch.reshape(arg_log, (probs.shape[0], 1))

            return torch.cat([x, logits_upd], dim=1)


    def save(self, path):
        torch.save({
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_seq_len': self.max_seq_len,
            'emb_size': self.emb_size,
            'num_heads': self.num_heads,
            'head_size': self.head_size,
            'num_layers': self.num_layers
        }, path)
    
    @classmethod
    def load(cls, path, device):
        checkpoint = torch.load(path, map_location=device)
        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_seq_len=checkpoint['max_seq_len'],
            emb_size=checkpoint['emb_size'],
            num_heads=checkpoint['num_heads'],
            head_size=checkpoint['head_size'],
            num_layers=checkpoint['num_layers']
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        return model

In [58]:
import glob

all_text = []
for file_path in glob.glob('./texts/*.*'):
    file = open(file_path, 'r', encoding='utf8')
    all_text.append(file.read())
    
all_text = '\n\n\n'.join(all_text)
len(all_text)

722690

In [59]:
bpe = BPE(2000)
bpe.fit(all_text)

160 | 2000
161 | 2000
162 | 2000
163 | 2000
164 | 2000
165 | 2000
166 | 2000
167 | 2000
168 | 2000
169 | 2000
170 | 2000
171 | 2000
172 | 2000
173 | 2000
174 | 2000
175 | 2000
176 | 2000
177 | 2000
178 | 2000
179 | 2000
180 | 2000
181 | 2000
182 | 2000
183 | 2000
184 | 2000
185 | 2000
186 | 2000
187 | 2000
188 | 2000
189 | 2000
190 | 2000
191 | 2000
192 | 2000
193 | 2000
194 | 2000
195 | 2000
196 | 2000
197 | 2000
198 | 2000
199 | 2000
200 | 2000
201 | 2000
202 | 2000
203 | 2000
204 | 2000
205 | 2000
206 | 2000
207 | 2000
208 | 2000
209 | 2000
210 | 2000
211 | 2000
212 | 2000
213 | 2000
214 | 2000
215 | 2000
216 | 2000
217 | 2000
218 | 2000
219 | 2000
220 | 2000
221 | 2000
222 | 2000
223 | 2000
224 | 2000
225 | 2000
226 | 2000
227 | 2000
228 | 2000
229 | 2000
230 | 2000
231 | 2000
232 | 2000
233 | 2000
234 | 2000
235 | 2000
236 | 2000
237 | 2000
238 | 2000
239 | 2000
240 | 2000
241 | 2000
242 | 2000
243 | 2000
244 | 2000
245 | 2000
246 | 2000
247 | 2000
248 | 2000
249 | 2000
250 | 2000

In [70]:
bpe.save('bpe1.bp')

Объект сохранён в bpe1.bp


In [71]:
print(bpe.id2token)
print(len(bpe.encode(all_text)))
print(len(bpe.decode(bpe.encode(all_text))))

{0: '\n', 1: ' ', 2: '!', 3: "'", 4: '(', 5: ')', 6: '*', 7: ',', 8: '-', 9: '.', 10: '/', 11: '0', 12: '1', 13: '2', 14: '3', 15: '4', 16: '5', 17: '6', 18: '7', 19: '8', 20: '9', 21: ':', 22: ';', 23: '<', 24: '>', 25: '?', 26: 'A', 27: 'B', 28: 'C', 29: 'D', 30: 'E', 31: 'F', 32: 'G', 33: 'H', 34: 'I', 35: 'J', 36: 'K', 37: 'L', 38: 'M', 39: 'N', 40: 'O', 41: 'P', 42: 'Q', 43: 'R', 44: 'S', 45: 'T', 46: 'U', 47: 'V', 48: 'W', 49: 'X', 50: 'Z', 51: '[', 52: ']', 53: 'a', 54: 'b', 55: 'c', 56: 'd', 57: 'e', 58: 'f', 59: 'g', 60: 'h', 61: 'i', 62: 'j', 63: 'k', 64: 'l', 65: 'm', 66: 'n', 67: 'o', 68: 'p', 69: 'q', 70: 'r', 71: 's', 72: 't', 73: 'u', 74: 'v', 75: 'w', 76: 'x', 77: 'y', 78: 'z', 79: '«', 80: '»', 81: 'à', 82: 'â', 83: 'ç', 84: 'è', 85: 'é', 86: 'ê', 87: 'î', 88: 'ï', 89: 'ô', 90: 'ù', 91: 'û', 92: '̀', 93: 'А', 94: 'Б', 95: 'В', 96: 'Г', 97: 'Д', 98: 'Е', 99: 'Ж', 100: 'З', 101: 'И', 102: 'Й', 103: 'К', 104: 'Л', 105: 'М', 106: 'Н', 107: 'О', 108: 'П', 109: 'Р', 110: 'С'

In [61]:
class GetData(Dataset):
    def __init__(self, data: list, seq_len: int, device: str='cuda'):
        self.data = data
        self.seq_len = seq_len
        self.device = device
    
    def __len__(self):
        return len(self.data) - self.seq_len -1
    
    def __getitem__(self, idx: int):
        x = torch.tensor(self.data[idx: idx+self.seq_len], dtype=torch.long, device=self.device)
        y = torch.tensor(self.data[idx+1: idx+self.seq_len], dtype=torch.long, device=self.device)

        return x, y

In [62]:
from torch.utils.data import DataLoader
import torch.optim as optim
from tqdm import tqdm

In [138]:
class GPT(nn.Module):
    def __init__(self, vocab_size: int, max_seq_len: int, emb_size: int, num_heads: int, head_size: int, num_layers: int, dropout: float=0.1, device: str='cuda'):
        super().__init__()

        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.head_size = head_size
        self.device = device
        self.num_layers = num_layers

        self.token_emb = TokenEmbeddings(vocab_size, emb_size)
        self.pos_emb = PositionalEmbeddings(max_seq_len, emb_size)
        self.drop = nn.Dropout(dropout)

        self.decoders = nn.Sequential(*[Decoder(num_heads, emb_size, head_size, max_seq_len, dropout) for _ in range(num_layers)])

        self.lin = nn.Linear(emb_size, vocab_size)

    def forward(self, x):

        x = x.to(self.device)

        x_embs = self.token_emb(x) + self.pos_emb(x.shape[1])


        x_dec = self.decoders(self.drop(x_embs))

        return self.lin(x_dec)
    
    def generate(self, x, max_new_tokens: int, do_sample: bool, temperature: float=1.0, top_k: int = None, top_p: float = None):
        for i in range(max_new_tokens):
            tokens = x[:, -self.max_seq_len:]
            logits = self.forward(tokens)[:, -1, :] / temperature

            if top_k and do_sample:
                last_logits = logits[:, -1, :]
                top_k_vals, _ = torch.sort(last_logits, dim=1, descending=True)
                min_value = top_k_vals[:, top_k-1].unsqueeze(1)
                last_logits[last_logits < min_value] = -float('inf')
            
            if top_p and do_sample:
                last_logits = logits[:, -1, :]
                probs = torch.softmax(last_logits, dim=1)
                probs_sort, idx_sort = torch.sort(probs, dim=1, descending=True)
                max_idx = idx_sort[:, 0].unsqueeze(1)
                arg_idx = torch.argsort(idx_sort, dim=1)
                cumsum = torch.cumsum(probs_sort, dim=1)
                cumsum_probs = torch.gather(cumsum, dim=1, index=arg_idx)
                mask = cumsum_probs > top_p
                mask.scatter_(dim=1, index=max_idx, value=False)
                last_logits[mask] = -float('inf')

            probs = torch.softmax(logits[:, -1, :], dim=1)

            if not do_sample:
                arg_log = torch.argmax(probs, dim=1)
            else:
                arg_log = torch.multinomial(probs, 1)

            logits_upd = torch.reshape(arg_log, (probs.shape[0], 1))

        return torch.cat([x, logits_upd], dim=1)

    def fit(self, train_loader, valid_loader, num_epoch, learning_rate):
        self.to(self.device)
        optimizer = optim.Adam(self.parameters(), learning_rate)
        criterion = nn.CrossEntropyLoss()

        for epoch in tqdm(range(num_epoch)):
            self.train()
            train_losses = []
            for inputs, targets in train_loader:
                inputs, targets = inputs.to(self.device), targets.to(self.device)

                logits = self.forward(inputs)[:, :-1, :]          # drop last position
                B, T, V = logits.shape
                logits_flat = logits.reshape(B * T, V)
                targets_flat = targets.reshape(B * T)

                loss = criterion(logits_flat, targets_flat)
                train_losses.append(loss.item())

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            avg_train_loss = sum(train_losses) / len(train_losses)

            # Validation
            self.eval()
            val_losses = []
            with torch.no_grad():
                for inputs, targets in valid_loader:
                    inputs, targets = inputs.to(self.device), targets.to(self.device)

                    logits = self.forward(inputs)[:, :-1, :]
                    B, T, V = logits.shape
                    logits_flat = logits.reshape(B * T, V)
                    targets_flat = targets.reshape(B * T)

                    loss = criterion(logits_flat, targets_flat)
                    val_losses.append(loss.item())

            avg_val_loss = sum(val_losses) / len(val_losses)
            print(f"Epoch {epoch}: Train Loss = {avg_train_loss:.4f}, Val Loss = {avg_val_loss:.4f}")
                    


    def save(self, path):
        torch.save({
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_seq_len': self.max_seq_len,
            'emb_size': self.emb_size,
            'num_heads': self.num_heads,
            'head_size': self.head_size,
            'num_layers': self.num_layers
        }, path)
    
    @classmethod
    def load(cls, path, device):
        checkpoint = torch.load(path, map_location=device)
        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_seq_len=checkpoint['max_seq_len'],
            emb_size=checkpoint['emb_size'],
            num_heads=checkpoint['num_heads'],
            head_size=checkpoint['head_size'],
            num_layers=checkpoint['num_layers']
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        return model

In [64]:
token_ids = bpe.encode(all_text)

In [65]:
n = int(0.8*len(token_ids))
train_token_ids = token_ids[:n]
valid_token_ids = token_ids[n:]

In [141]:
num_layers = 12
num_heads = 8
head_size = 64
emb_size = 128
vocab_size = 2000
dropout= 0.1
learning_rate = 0.1
num_epochs = 5
batch_size = 512
seq_len = 64
max_seq_len = 256
device = 'cpu'

In [122]:
train_dataset = GetData(train_token_ids, seq_len, device)
train_loader = DataLoader(train_dataset, batch_size)

valid_dataset = GetData(valid_token_ids, seq_len, device)
valid_loader = DataLoader(valid_dataset, batch_size)

In [142]:
gpt = GPT(vocab_size, max_seq_len, emb_size, num_heads, head_size, num_layers, device='cpu')
gpt.save('m1.pth')

In [ ]:
gpt.fit(train_loader, valid_loader, num_epochs, learning_rate)

 20%|██        | 1/5 [27:08<1:48:35, 1628.82s/it]

Epoch 0: Train Loss = 6.6690, Val Loss = 6.4527


 40%|████      | 2/5 [56:12<1:24:48, 1696.20s/it]

Epoch 1: Train Loss = 6.0179, Val Loss = 6.2675


 60%|██████    | 3/5 [1:25:54<57:51, 1735.50s/it]

Epoch 2: Train Loss = 5.9264, Val Loss = 6.4585


 80%|████████  | 4/5 [1:55:00<28:59, 1739.69s/it]

Epoch 3: Train Loss = 5.7941, Val Loss = 6.3383
